### Question 1

In [1]:
from embedder import Embedder
embedder = Embedder()

In [2]:
q1 = "How does approximate nearest neighbor search work?"
encode_text = embedder.encode(q1)

In [3]:
len(encode_text)

384

In [4]:
encode_text[0]

np.float64(-0.02058203437252893)

### Question 2

### LOADING DATA

In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [6]:
#check the document
documents[:5]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [7]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])

X = embedder.encode_batch(
    [doc["content"] for doc in documents]
)

vindex.fit(X, documents)

In [8]:
# sqlitesearch-vecto
query_vector = embedder.encode(
    "How does approximate nearest neighbor search work?"
)

results = vindex.search(
    query_vector=query_vector,
    filter_dict={
        "filename": "02-vector-search/lessons/07-sqlitesearch-vector.md"
                 },
    num_results=5
)

In [9]:
content_q2  = results[0]['content']
doc_vector = embedder.encode(content_q2)

query_vector.dot(doc_vector)


np.float64(0.36107027225589694)

### Question 3

In [10]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks generated: {len(chunks)}")

Total chunks generated: 295


In [11]:
from minsearch import VectorSearch

vindex_chunk = VectorSearch(keyword_fields=["filename"])

X = embedder.encode_batch(
    [doc["content"] for doc in chunks]
)

vindex_chunk.fit(X, chunks)

### Check each of filename

#### Part 1

In [12]:
query_vector = embedder.encode(
    "How does approximate nearest neighbor search work?"
)

results_q31 = vindex_chunk.search(
    query_vector=query_vector,
    filter_dict={
        "filename": "02-vector-search/lessons/03-embeddings-dataset.md"
                 },
    num_results=5
)
content_q31  = results_q31[0]['content']
chunk_vector1 = embedder.encode(content_q31)

query_vector.dot(chunk_vector1)

np.float64(0.10629072774015885)

#### Part 2

In [13]:
query_vector = embedder.encode(
    "How does approximate nearest neighbor search work?"
)

results_q32 = vindex_chunk.search(
    query_vector=query_vector,
    filter_dict={
        "filename": "02-vector-search/lessons/06-rag-vector.md"
                 },
    num_results=5
)

content_q32  = results_q32[0]['content']
chunk_vector2 = embedder.encode(content_q32)

query_vector.dot(chunk_vector2)

np.float64(0.3328189994383671)

#### Part 3

In [14]:
query_vector = embedder.encode(
    "How does approximate nearest neighbor search work?"
)

results_q33 = vindex_chunk.search(
    query_vector=query_vector,
    filter_dict={
        "filename": "02-vector-search/lessons/07-sqlitesearch-vector.md"
                 },
    num_results=5
)

content_q33  = results_q33[0]['content']
chunk_vector3 = embedder.encode(content_q33)

query_vector.dot(chunk_vector3)

np.float64(0.6489017718578812)

#### Part 4

In [15]:
query_vector = embedder.encode(
    "How does approximate nearest neighbor search work?"
)

results_q34 = vindex_chunk.search(
    query_vector=query_vector,
    filter_dict={
        "filename": "02-vector-search/lessons/09-onnx-embedder.md"
                 },
    num_results=5
)

content_q34  = results_q34[0]['content']
chunk_vector4 = embedder.encode(content_q34)

query_vector.dot(chunk_vector4)

np.float64(0.18088101411723273)

So the highest score is when we use keyword 
"02-vector-search/lessons/07-sqlitesearch-vector.md" 
where the score ~ 0.64

### Question 4

In [ ]:
# sqlitesearch-vecto
query_vector_q4 = embedder.encode(
    "What metric do we use to evaluate a search engine?"
)

results_q4 = vindex.search(
    query_vector=query_vector_q4,
    num_results=5
)

results_q4[0]

{'content': '# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup, each query 

### Question 5

#### Vector Results

In [17]:
query_vector_q5 = embedder.encode(
    "How do I store vectors in PostgreSQL?"
)

filename_check = ["02-vector-search/lessons/01-intro.md",
                  "02-vector-search/lessons/02-embeddings.md",
                  "02-vector-search/lessons/08-pgvector.md",
                  "03-orchestration/lessons/05-rag.md"]

check_result = []
for i in filename_check:
    results_q5 = vindex_chunk.search(
        query_vector=query_vector_q5,
            filter_dict={
            "filename": i
                    },
        num_results=5
    )
    check_result.append(results_q5[0])


In [18]:
check_result

[{'start': 2000,
  'content': "ch uses an inverted index (BM25, TF-IDF). Vector search\n  uses a vector index based on cosine similarity.\n- Keyword search misses synonyms and paraphrases. Vector search misses\n  exact term matches.\n\nVector search is usually better, but it adds a lot of operational\ncomplexity, and you'll feel that throughout this module. So my advice\nis to never start with vector search. Start with text search, and reach\nfor vectors once you can show they're worth the extra cost.\n\nIn practice the two work best together. Hybrid search combines them,\nand we cover it in the\n[Best Practices module](../../06-best-practices/lessons/02-hybrid-search.md).\n\n## Building vector search\n\nWe'll take the same FAQ dataset from module 1 and build vector search\nwith three tools:\n\n1. minsearch - in-memory vector search (simplest, good for\n   experiments)\n2. sqlitesearch - persistent vector search backed by SQLite\n   (production-friendly, same API as minsearch)\n3. PGVe

#### all filename options are available if check with vector search

#### Text Results

In [19]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

question = "How do I store vectors in PostgreSQL?"
text_result = index.search(question, num_results=5)
text_result

[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

#### Based on the results, 
filename "02-vector-search/lessons/08-pgvector.md" is not found in text search 

### Question 6

In [20]:
#define hybrid search function
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"]) # exclude part , doc["start"] because in document, there are not containing start key
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

#### find out text result and vector result

In [21]:
query = "How do I give the model access to tools?"
query_vector = embedder.encode(
    query
)

## vector result
vector_results = vindex.search(
            query_vector=query_vector,
            num_results=5
        )

text_results = index.search(
            query, 
            num_results=5
        )

results = rrf([vector_results, text_results])

In [22]:
results

[{'start': 1000,
  'content': 'decides when to call it and what to search for.\n\nThe same typo question now goes like this:\n\n```mermaid\nflowchart TD\n    U([User: How do I run Olama?])\n    L1[LLM: I\'ll search for \'Olama\']\n    S1[search - Olama - no useful results]\n    L2[LLM: Hmm, no results. Maybe a typo for \'Ollama\'?]\n    S2[search - Ollama - found results!]\n    A([LLM: Here\'s how to run Ollama locally...])\n\n    U --> L1 --> S1 --> L2 --> S2 --> A\n```\n\nThe LLM searched, saw the results were bad, and decided to try again\nwith a different query. It made that decision on its own. We didn\'t\nwrite any code to handle typos.\n\nThe difference is about who makes the decisions:\n\n- With RAG, the developer decides. We fix the steps up front, so\n  search always runs once with the exact user query.\n- With an agent, the LLM decides. It chooses which actions to take\n  and when to stop.\n\nThe mechanism that makes this possible is function calling, and that\'s\nwhat the r

In [23]:
results[0]

{'start': 1000,
 'content': 'decides when to call it and what to search for.\n\nThe same typo question now goes like this:\n\n```mermaid\nflowchart TD\n    U([User: How do I run Olama?])\n    L1[LLM: I\'ll search for \'Olama\']\n    S1[search - Olama - no useful results]\n    L2[LLM: Hmm, no results. Maybe a typo for \'Ollama\'?]\n    S2[search - Ollama - found results!]\n    A([LLM: Here\'s how to run Ollama locally...])\n\n    U --> L1 --> S1 --> L2 --> S2 --> A\n```\n\nThe LLM searched, saw the results were bad, and decided to try again\nwith a different query. It made that decision on its own. We didn\'t\nwrite any code to handle typos.\n\nThe difference is about who makes the decisions:\n\n- With RAG, the developer decides. We fix the steps up front, so\n  search always runs once with the exact user query.\n- With an agent, the LLM decides. It chooses which actions to take\n  and when to stop.\n\nThe mechanism that makes this possible is function calling, and that\'s\nwhat the res